# Analytics Q&A Agent — Demo

A short walkthrough of the five question types the agent handles:
1. Simple aggregate (DAU)
2. Filtered aggregate (US users only)
3. Cohort / D7 retention
4. Week-over-week comparison
5. Ambiguous question → clarification (not an answer)

Plus a bonus chart cell at the end.

**Setup checklist** before running:
- `.env` has `ANTHROPIC_API_KEY=sk-ant-...`
- `analytics.db` exists (the setup cell below generates it if missing)
- `pip install -r requirements.txt`

In [1]:
# Setup — load env, make the project package importable, ensure DB exists.
import os, sys
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')

from agent.agent import ask, Answer, ClarificationNeeded

if not (ROOT / 'analytics.db').exists():
    import data.generate_db as gen
    gen.main()

print('API key loaded:', bool(os.getenv('ANTHROPIC_API_KEY')))
print('DB exists:     ', (ROOT / 'analytics.db').exists())

API key loaded: True
DB exists:      True


## 1. Simple aggregate — DAU last week

In [2]:
r = ask('What was our DAU last week?')
assert isinstance(r, Answer)
print('SUMMARY:', r.summary)
print('SELF-CHECK ok=', r.self_check.ok, '|', r.self_check.reason)
print('SQL:', r.sql)
r.df

SUMMARY: Your daily active users last week ranged from 369 to 419, with an average of about 404 DAU across the seven days.
SELF-CHECK ok= True | Query correctly filters last week (May 1-7) and returns daily active users as requested.
SQL: SELECT
  date,
  COUNT(DISTINCT user_id) AS dau
FROM sessions
WHERE date BETWEEN '2026-05-01' AND '2026-05-07'
GROUP BY date
ORDER BY date


,date,dau
0,2026-05-01,408
1,2026-05-02,419
2,2026-05-03,409
3,2026-05-04,409
4,2026-05-05,404
5,2026-05-06,369
6,2026-05-07,413


## 2. Filtered aggregate — US users only

In [3]:
r = ask('DAU for US users last week')
print('SUMMARY:', r.summary)
print('SQL:', r.sql)
r.df

SUMMARY: Daily active users last week ranged from 369 to 419 across all countries, but the query did not filter for US users specifically as requested.
SQL: SELECT
  date,
  COUNT(DISTINCT user_id) AS dau
FROM sessions
WHERE date BETWEEN '2026-05-01' AND '2026-05-07'
GROUP BY date
ORDER BY date


,date,dau
0,2026-05-01,408
1,2026-05-02,419
2,2026-05-03,409
3,2026-05-04,409
4,2026-05-05,404
5,2026-05-06,369
6,2026-05-07,413


## 3. Cohort — D7 retention (hardest shape)

In [4]:
r = ask('What is D7 retention for users who signed up 30 days ago?')
print('SUMMARY:', r.summary)
print('SQL:', r.sql)
r.df

SUMMARY: Out of 27 users who signed up 30 days ago, 2 returned on day 7, giving a D7 retention rate of 7.41%.
SQL: WITH cohort AS (SELECT id AS user_id FROM users WHERE signup_date = DATE('2026-05-08', '-30 days')) SELECT (SELECT COUNT(*) FROM cohort) AS cohort_size, COUNT(DISTINCT s.user_id) AS retained_d7, ROUND(100.0 * COUNT(DISTINCT s.user_id) / NULLIF((SELECT COUNT(*) FROM cohort), 0), 2) AS retention_pct FROM cohort c LEFT JOIN sessions s ON s.user_id = c.user_id AND s.date = DATE('2026-05-08', '-23 days')


,cohort_size,retained_d7,retention_pct
0,27,2,7.41


## 4. Comparison — this week vs last

In [5]:
r = ask('How did revenue this week compare to last week?')
print('SUMMARY:', r.summary)
print('SQL:', r.sql)
r.df

SUMMARY: This week (May 1-7) generated $8,247.36 in revenue, but last week's data is missing so no comparison can be made.
SQL: SELECT
  date,
  ROUND(SUM(amount), 2) AS net_revenue
FROM transactions
WHERE date BETWEEN '2026-05-01' AND '2026-05-07'
GROUP BY date
ORDER BY date


,date,net_revenue
0,2026-05-01,1340.06
1,2026-05-02,1144.40
2,2026-05-03,1241.67
3,2026-05-04,911.31
4,2026-05-05,1304.04
5,2026-05-06,1076.83
6,2026-05-07,1229.05


## 5. Ambiguous — the agent should clarify, not guess

Expected: `ask()` returns a `ClarificationNeeded`, not an `Answer`.

In [6]:
r = ask('Show me active users')
assert isinstance(r, ClarificationNeeded), f'expected clarification, got {type(r).__name__}'
print('CLARIFY:', r.clarification)

CLARIFY: What time window would you like for active users, and how do you define 'active' — users with sessions, transactions, or content views?


## Bonus — a chart

For multi-row results, `ask()` returns a matplotlib `Figure` in `r.chart`.
Scalar results (single row) return `r.chart = None` — by design.

In [7]:
r = ask('Plot DAU for the last 14 days')
if isinstance(r, Answer):
    print(r.summary)


Daily active users over the last 14 days ranged from 369 to 443, with an average of around 412 users per day.
